# 01 · Datos

Construye los 14 esquemas propios (12 de entrenamiento, 2 de validación), genera los pares
pregunta–SQL y **valida cada consulta en MySQL 8.4.10** (configuración por defecto, con
`ONLY_FULL_GROUP_BY`). Solo se aceptan pares que corren sin error; los descartes quedan en
`datos/descartes/descartes.csv` con su motivo.

Todo es determinista (semilla 2026): correrlo de nuevo produce exactamente los mismos archivos.

In [ ]:
# --- Preparar el entorno (Colab) ---
# Opción 1: clonar el repo privado con un token de GitHub guardado en los "Secrets" de Colab (GITHUB_TOKEN)
import os
REPO = "USUARIO/proyecto3-text2sql"   # <-- cambiar por el repo de la pareja
if not os.path.exists("/content/proyecto3-text2sql"):
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    !git clone -q https://{token}@github.com/{REPO}.git /content/proyecto3-text2sql
%cd /content/proyecto3-text2sql
!pip install -q -r requirements.txt
!bash scripts/instalar_mysql_colab.sh

## 1. Esquemas: datos sintéticos + `SHOW CREATE TABLE`

In [ ]:
!python datos/construir_esquemas.py --socket /tmp/mysql.sock --crear-lector

In [ ]:
print(open('datos/esquemas/val/gym_mgmt.schema.sql').read()[:1500])

## 2. Pares de entrenamiento validados en MySQL

In [ ]:
!python datos/validar_pares.py --socket /tmp/mysql.sock

In [ ]:
import pandas as pd, json
tr = pd.read_json("datos/entrenamiento/train.jsonl", lines=True)
print(len(tr), "pares")
display(tr.groupby(["db", "nivel"]).size().unstack(fill_value=0))
display(tr.sample(8, random_state=1)[["db", "nivel", "pregunta", "sql"]])

## 3. Descartes

In [ ]:
d = pd.read_csv("datos/descartes/descartes.csv")
display(d.groupby(["motivo"]).size().rename("cantidad"))
display(d[d.motivo != "duplicado"].head(10)[["db", "motivo", "detalle", "sql"]])

## 4. Ejemplos de reparación (consulta rota + error real de MySQL)

In [ ]:
!python datos/generar_reparaciones.py --socket /tmp/mysql.sock

In [ ]:
rep = pd.read_json("datos/entrenamiento/reparaciones.jsonl", lines=True)
display(rep.groupby("error_tipo").size())
display(rep.head(5)[["sql_erronea", "error", "sql"]])

## 5. Verificar la partición por esquema (penalización de −15 si se mezcla)

In [ ]:
import sys; sys.path.insert(0, "src")
import os; os.environ["T2SQL_SOLO_PROMPT"] = "1"
import datos_chat
p = datos_chat.verificar_particion()
print(p)
assert not p["tablas_compartidas"], "hay tablas de validación en entrenamiento"

In [ ]:
# Validación propia: las 40 consultas de referencia corren y devuelven filas
import pymysql
val = datos_chat.preguntas_validacion()
con = pymysql.connect(unix_socket="/tmp/mysql.sock", user="root", charset="utf8mb4")
with con.cursor() as cur:
    for q in val:
        cur.execute(f"USE {q['db']}"); cur.execute(q["sql"]); n = len(cur.fetchall())
        assert n > 0, q["id"]
print(pd.Series([q["nivel"] for q in val]).value_counts())